<div style="background-color:#e2725b; padding: 18px; border-radius: 3px; text-align:center; font-size:2.5em; font-weight:bold; color:#222; margin-bottom:25px; letter-spacing:1px;">
Predicting Donor Response for Social Good 
</div>

# <h2 style="border-bottom: 4px solid #e2725b; width:fit-content; margin: 0 auto 25px auto; padding-bottom:6px; font-weight:bold;">Baseline Models</h2>

_Data Mining II 2025/2026_

Project by:
Francisco Gomes (20221810), Margarida Marchão (20221901), Marta Alves (20221890), Pedro Coimbras (20211573)


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">1. Imports and data loading</h3>

This notebook establishes the first modeling baseline for the donor-response problem. The objective at this stage is not to maximize performance yet, but to create a clean and reliable workflow that respects the findings from the EDA.


In [35]:
"""Importing the libraries needed for data handling, preprocessing, modeling, and evaluation"""
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

"""Adding the project root to the Python path so notebook imports work correctly"""
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


In [36]:
"""Loading the training dataset for baseline modeling"""
data = pd.read_csv(PROJECT_ROOT / "project_data" / "donors_train.csv", index_col=0)
data.head()


,CARD_PROM_12,CHILDREN,DONOR_AGE,DONOR_GENDER,FILE_CARD_GIFT,FREQUENCY_STATUS_97NK,HOME_OWNER,INCOME_GROUP,LAST_GIFT_AMT,LIFETIME_CARD_PROM,LIFETIME_GIFT_AMOUNT,LIFETIME_GIFT_COUNT,LIFETIME_MAX_GIFT_AMT,LIFETIME_MIN_GIFT_AMT,LIFETIME_PROM,MEDIAN_HOME_VALUE,MEDIAN_HOUSEHOLD_INCOME,MONTHS_SINCE_FIRST_GIFT,MONTHS_SINCE_LAST_GIFT,MONTHS_SINCE_LAST_PROM_RESP,NUMBER_PROM_12,PCT_ATTRIBUTE1,PCT_ATTRIBUTE2,PCT_ATTRIBUTE3,PCT_ATTRIBUTE4,PCT_OWNER_OCCUPIED,PEP_STAR,PER_CAPITA_INCOME,RECENCY_STATUS_96NK,RECENT_AVG_CARD_GIFT_AMT,RECENT_AVG_GIFT_AMT,RECENT_CARD_RESPONSE_COUNT,RECENT_CARD_RESPONSE_PROP,RECENT_RESPONSE_COUNT,RECENT_RESPONSE_PROP,RECENT_STAR_STATUS,SES,URBANICITY,WEALTH_RATING,TARGET_B
CONTROL_NUMBER,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
61745,4.0,3.0,33.0,M,0.0,1.0,H,5.0,20.0,9.0,35.0,2.0,20.0,15.0,21.0,566.0,315.0,182.037132,18.0,18.0,10.0,0.0,52.0,17.0,25.0,92.0,0.0,12827.0,A,0.00,17.50,NaN,0.000,2.0,0.154,0.0,2,T,NaN,1
112703,3.0,2.0,NaN,F,1.0,1.0,U,NaN,15.0,6.0,15.0,1.0,15.0,15.0,15.0,318.0,148.0,24.000000,24.0,24.0,7.0,0.0,31.0,31.0,39.0,73.0,0.0,7787.0,N,15.00,15.00,1.0,0.250,1.0,0.100,0.0,3,R,NaN,1
166437,4.0,2.0,NaN,F,7.0,3.0,H,4.0,10.0,17.0,79.0,11.0,12.0,5.0,40.0,1669.0,373.0,129.000000,15.0,15.0,8.0,0.0,26.0,39.0,38.0,84.0,1.0,13965.0,S,0.00,10.67,0.0,0.000,3.0,0.231,1.0,1,U,NaN,0
170621,4.0,NaN,61.0,M,13.0,1.0,H,6.0,11.0,28.0,80.0,17.0,11.0,3.0,75.0,1464.0,488.0,130.000000,16.0,16.0,13.0,0.0,48.0,30.0,44.0,84.0,1.0,24123.0,A,10.00,10.00,2.0,0.286,2.0,0.111,0.0,1,U,NaN,0
44428,6.0,0.0,75.0,M,3.0,4.0,H,3.0,7.0,9.0,27.0,5.0,7.0,5.0,22.0,936.0,249.0,24.000000,17.0,17.0,13.0,0.0,52.0,NaN,66.0,90.0,1.0,15008.0,N,5.67,5.40,3.0,0.600,5.0,0.500,0.0,2,C,NaN,0


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2. Problem setup and data partition</h3>

Following good modeling practice, the first step is to separate features and target and create a train/validation split before any fitted preprocessing is applied. This helps avoid data leakage and makes the baseline evaluation more reliable.


In [37]:
"""Defining the target and identifier variables and creating the train-validation split"""
TARGET = "TARGET_B"
ID_COLUMN = "CONTROL_NUMBER"
RANDOM_STATE = 5
TRAIN_SIZE = 0.70

X = data.drop(columns=[TARGET])
y = data[TARGET].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    train_size=TRAIN_SIZE,
    shuffle=True,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Training target rate: {y_train.mean():.3f}")
print(f"Validation target rate: {y_val.mean():.3f}")


Training set shape: (9492, 39)
Validation set shape: (4068, 39)
Training target rate: 0.250
Validation target rate: 0.250


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">3. Rule-based cleaning from the EDA</h3>

At this stage, we only apply the cleaning rules that were directly supported by the EDA. These are deterministic corrections, not learned preprocessing steps, so they can be applied consistently to both the training and validation sets.


In [38]:
"""Creating a rule-based cleaning function using only the issues already justified in the EDA"""
def apply_eda_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """Apply only the domain rules already justified in the EDA notebook."""
    cleaned = df.copy()

    """Treating ambiguous categorical codes as missing values"""
    for column in ["SES", "URBANICITY"]:
        if column in cleaned.columns:
            cleaned[column] = cleaned[column].replace("?", np.nan)

    """Converting domain-invalid numeric values into missing values"""
    if "CHILDREN" in cleaned.columns:
        cleaned.loc[cleaned["CHILDREN"] < 0, "CHILDREN"] = np.nan

    if "RECENT_RESPONSE_PROP" in cleaned.columns:
        invalid_recent_response = (
            (cleaned["RECENT_RESPONSE_PROP"] < 0)
            | (cleaned["RECENT_RESPONSE_PROP"] > 1)
        )
        cleaned.loc[invalid_recent_response, "RECENT_RESPONSE_PROP"] = np.nan

    if "RECENT_CARD_RESPONSE_PROP" in cleaned.columns:
        invalid_recent_card_response = (
            (cleaned["RECENT_CARD_RESPONSE_PROP"] < 0)
            | (cleaned["RECENT_CARD_RESPONSE_PROP"] > 1)
        )
        cleaned.loc[invalid_recent_card_response, "RECENT_CARD_RESPONSE_PROP"] = np.nan

    return cleaned


In [39]:
"""Applying the EDA-based cleaning rules to the training and validation partitions and summarizing the detected issues"""
X_train_clean = apply_eda_cleaning(X_train)
X_val_clean = apply_eda_cleaning(X_val)

cleaning_summary = pd.DataFrame(
    {
        "invalid_children_train": [int((X_train["CHILDREN"] < 0).sum())],
        "invalid_recent_response_prop_train": [
            int(((X_train["RECENT_RESPONSE_PROP"] < 0) | (X_train["RECENT_RESPONSE_PROP"] > 1)).sum())
        ],
        "invalid_recent_card_response_prop_train": [
            int(((X_train["RECENT_CARD_RESPONSE_PROP"] < 0) | (X_train["RECENT_CARD_RESPONSE_PROP"] > 1)).sum())
        ],
        "ses_question_mark_train": [int((X_train["SES"].astype(str) == "?").sum())],
        "urbanicity_question_mark_train": [int((X_train["URBANICITY"].astype(str) == "?").sum())],
    }
)

display(cleaning_summary)


,invalid_children_train,invalid_recent_response_prop_train,invalid_recent_card_response_prop_train,ses_question_mark_train,urbanicity_question_mark_train
0,48,46,51,221,223


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">4. Feature groups and preprocessing design</h3>

The baseline preprocessing should remain simple. Numerical variables will be imputed with the median and scaled, while categorical variables will be imputed with a constant label and one-hot encoded. The identifier remains outside the modeling matrix because it is loaded as the dataframe index.


In [40]:
"""Preparing the modeling matrices and separating numeric and categorical features"""
X_train_model = X_train_clean.copy()
X_val_model = X_val_clean.copy()

numeric_features = X_train_model.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train_model.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Number of numeric features: {len(numeric_features)}")
print(f"Number of categorical features: {len(categorical_features)}")
print("Categorical features:", categorical_features)


Number of numeric features: 34
Number of categorical features: 5
Categorical features: ['DONOR_GENDER', 'HOME_OWNER', 'RECENCY_STATUS_96NK', 'SES', 'URBANICITY']


In [41]:
"""Building the preprocessing objects for the baseline models"""
def build_one_hot_encoder() -> OneHotEncoder:
    """Create a one-hot encoder compatible with different scikit-learn versions."""
    params = {"handle_unknown": "ignore"}
    if "sparse_output" in OneHotEncoder.__init__.__code__.co_varnames:
        params["sparse_output"] = False
    else:
        params["sparse"] = False
    return OneHotEncoder(**params)

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("encoder", build_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
)


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">5. Baseline model definitions</h3>

The first comparison focuses on simple and widely used baseline classifiers. The goal is to understand which model family looks most promising before moving to feature engineering and model selection.


In [42]:
"""Defining the first set of baseline classifiers to compare"""
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE,
    ),
}

models


{'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=5),
 'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                        random_state=5),
 'Gradient Boosting': GradientBoostingClassifier(random_state=5)}

In [43]:
"""Creating a helper function to train a model and evaluate it on the validation set"""
def evaluate_classifier(model_pipeline, X_train, X_val, y_train, y_val) -> dict:
    model_pipeline.fit(X_train, y_train)

    y_pred = model_pipeline.predict(X_val)
    y_proba = model_pipeline.predict_proba(X_val)[:, 1]

    return {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, y_proba),
    }


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">6. Baseline training and validation results</h3>

Since the official competition metric has not yet been fixed in this notebook, we compare models using a small set of standard classification metrics. For ranking the first baselines, `ROC-AUC` is used as the main reference metric.


In [44]:
"""Training each baseline pipeline and comparing the validation metrics"""
results = []
fitted_pipelines = {}

for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", estimator),
        ]
    )

    metrics = evaluate_classifier(
        model_pipeline=pipeline,
        X_train=X_train_model,
        X_val=X_val_model,
        y_train=y_train,
        y_val=y_val,
    )

    results.append({"model": model_name, **metrics})
    fitted_pipelines[model_name] = pipeline

baseline_results = (
    pd.DataFrame(results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

baseline_results.style.format(
    {
        "accuracy": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}",
        "roc_auc": "{:.3f}",
    }
)


,model,accuracy,precision,recall,f1,roc_auc
0,Gradient Boosting,0.751,0.528,0.037,0.070,0.603
1,Random Forest,0.751,0.567,0.017,0.032,0.598
2,Logistic Regression,0.576,0.306,0.549,0.393,0.588


In [45]:
"""Displaying the model with the best validation ROC-AUC"""
best_model_name = baseline_results.loc[0, "model"]
print(f"Best baseline model by ROC-AUC: {best_model_name}")


Best baseline model by ROC-AUC: Gradient Boosting


> **Main Insights**  
This baseline comparison gives us a first reference point for model quality under a clean and leakage-safe workflow. At this stage, the goal is not yet to optimize every detail, but rather to identify which model family responds best to the current feature set and preprocessing choices.

**Interpretation.**  
The best-performing baseline can now be used as the benchmark for the next stage of the project. From here, we can test whether feature engineering improves performance and later move into more formal cross-validation and model tuning.
